In [5]:
from os import remove
from xml.dom.minidom import Document

from langchain_classic.chains.summarize.refine_prompts import prompt_template
from pandas.conftest import lexsorted_two_level_string_multiindex
from tensorflow.python.ops.gen_tpu_ops import retrieve_tpu_embedding_rms_prop_parameters

from Learn_GenAi_Tutorial_youtube.Agent_Tools.google_search_AI_Agent import question
from Learn_GenAi_Tutorial_youtube.Agent_Tools.project_4 import answer
from Learn_GenAi_Tutorial_youtube.Langchain_component_RAG_PIPELINE.vector_embedding.vector_embedding import documents, \
    embedding, vector_store
from tutedude_Classes_assigment.Assignment_22_Embedding_Models_vector_similarity_search.Embedding_models import chunks
# we build RAG application which communicate with this app to get to know or to get solve our query with respect to youtube video
#
# Matlab: Pehle video ka text nikaalenge. Phir us text ko chhote chunks mein divide karenge, embeddings banayenge aur FAISS mein store karenge. User ka question aane par relevant chunks retrieve honge. LLM un chunks ko use karke answer dega.


!pip install youtube-transcript-api
%pip install -U youtube-transcript-api langchain-text-splitters langchain-community langchain-huggingface langchain-groq faiss-cpu python-dotenv



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Part 2: Technologies we will use
#
# | Technology               | Why are we using it?                                         |
# | ------------------------ | ------------------------------------------------------------ |
# | Python                   | Application ki programming ke liye                           |
# | `youtube-transcript-api` | YouTube video ka available transcript retrieve karne ke liye |
# | LangChain                | RAG ke components ko connect karne ke liye                   |
# | Hugging Face Embeddings  | Text ko numerical vectors mein convert karne ke liye         |
# | FAISS                    | Similarity search aur relevant chunks retrieve karne ke liye |
# | Groq LLM                 | Retrieved context se answers generate karne ke liye          |
# | Streamlit                | Chatbot ka user interface banane ke liye                     |


In [1]:
!pip install -U langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers youtube-transcript-api streamlit python-dotenv

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached faiss_cpu-1.15.1-cp310-abi3-macosx_14_0_arm64.whl.metadata (7.8 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)
Using cached faiss_cpu-1.15.1-cp310-abi3-macosx_14_0_arm64.whl (5.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 3.4 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 3.7 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 3.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 3.8 MB/s  0:00:34m0:00:0100:02
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.1
    Uninstalling python-dote

In [7]:

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv()

/var/folders/jt/5qrgtw_s7858wm8jx1mwyygw0000gn/T/ipykernel_93059/874724680.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


False

In [45]:
load_dotenv()

True

In [10]:
# Document loader
youtube_api = YouTubeTranscriptApi()
video_id = 'Q3zwkxqh1t0'
transcripts = youtube_api.fetch(video_id)

In [11]:
transcripts

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='♪ "Stampede"\nby Genesis Owusu ♪', start=0.056, duration=3.151), FetchedTranscriptSnippet(text="Once you've been to the moon...", start=3.207, duration=1.393), FetchedTranscriptSnippet(text='♪', start=4.6, duration=1.193), FetchedTranscriptSnippet(text='[WHOOSHING]\n...where do you go next?', start=5.793, duration=1.335), FetchedTranscriptSnippet(text='♪', start=7.128, duration=1.585), FetchedTranscriptSnippet(text='Back to work.', start=8.713, duration=1.084), FetchedTranscriptSnippet(text='♪', start=9.797, duration=1.87), FetchedTranscriptSnippet(text='[CLICKING]', start=11.667, duration=1.62), FetchedTranscriptSnippet(text='♪', start=13.287, duration=1.849), FetchedTranscriptSnippet(text='You start with the chip.', start=15.136, duration=1.585), FetchedTranscriptSnippet(text='You engineer a new CPU core.', start=16.721, duration=2.627), FetchedTranscriptSnippet(text='You engineer a new GPU core.', start=19.348, duration=2.50

In [15]:
# remove() seprated bytes , text
transcripts_text = " ".join(doc.text for doc in transcripts)

In [16]:
transcripts_text


'♪ "Stampede"\nby Genesis Owusu ♪ Once you\'ve been to the moon... ♪ [WHOOSHING]\n...where do you go next? ♪ Back to work. ♪ [CLICKING] ♪ You start with the chip. You engineer a new CPU core. You engineer a new GPU core. You add a second Neural Engine. Now you\'ve just doubled\nthe compute power, pushed graphics\nup to 40 percent faster, unlocked Siri AI, and built the fastest chip\never put in an iPhone. ♪ But there\'s a catch: More power creates more heat. So you give the vapor chamber\nthree times more surface area, because a bigger vapor chamber\ncools the chip. A cooler chip creates\na more efficient chip. A more efficient chip\nuses less battery. ♪ A battery that works less\nlasts longer. ♪ The longest battery life\nin iPhone history. ♪ [CLICK] Next, you strengthen it... [CLICK]\n[THUD] [HISSING]\n...and test it... ♪ ...for wherever you need\nto take it. Then you ask, "What\'s next\nfor the camera system?" You want to improve\nlow-light photography and depth of field,\nso you des

In [17]:
# text splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=50)
splitter.split_text(transcripts_text)

['♪ "Stampede"\nby Genesis Owusu ♪ Once you\'ve been to the moon... ♪ [WHOOSHING]',
 '...where do you go next? ♪ Back to work. ♪ [CLICKING] ♪ You start with the chip. You engineer a new',
 '♪ You start with the chip. You engineer a new CPU core. You engineer a new GPU core. You add a',
 "CPU core. You engineer a new GPU core. You add a second Neural Engine. Now you've just doubled",
 'the compute power, pushed graphics',
 'up to 40 percent faster, unlocked Siri AI, and built the fastest chip',
 "ever put in an iPhone. ♪ But there's a catch: More power creates more heat. So you give the vapor",
 'power creates more heat. So you give the vapor chamber',
 'three times more surface area, because a bigger vapor chamber\ncools the chip. A cooler chip creates',
 'cools the chip. A cooler chip creates\na more efficient chip. A more efficient chip',
 'a more efficient chip. A more efficient chip\nuses less battery. ♪ A battery that works less',
 'uses less battery. ♪ A battery that works less\n

In [20]:
chunks = splitter.split_text(transcripts_text)

In [21]:
chunks

['♪ "Stampede"\nby Genesis Owusu ♪ Once you\'ve been to the moon... ♪ [WHOOSHING]',
 '...where do you go next? ♪ Back to work. ♪ [CLICKING] ♪ You start with the chip. You engineer a new',
 '♪ You start with the chip. You engineer a new CPU core. You engineer a new GPU core. You add a',
 "CPU core. You engineer a new GPU core. You add a second Neural Engine. Now you've just doubled",
 'the compute power, pushed graphics',
 'up to 40 percent faster, unlocked Siri AI, and built the fastest chip',
 "ever put in an iPhone. ♪ But there's a catch: More power creates more heat. So you give the vapor",
 'power creates more heat. So you give the vapor chamber',
 'three times more surface area, because a bigger vapor chamber\ncools the chip. A cooler chip creates',
 'cools the chip. A cooler chip creates\na more efficient chip. A more efficient chip',
 'a more efficient chip. A more efficient chip\nuses less battery. ♪ A battery that works less',
 'uses less battery. ♪ A battery that works less\n

In [18]:
embedding = HuggingFaceEmbeddings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [22]:
vector_store = FAISS.from_texts(chunks,embedding)

In [23]:
vector_store

In [24]:
vector_store.similarity_search(query='what is feature in iphone 18 pro')

[Document(id='c686c5ed-f36b-4a2b-bfd2-31f213c64774', metadata={}, page_content='to give photographers more creative control. So you add pro controls, allowing them to quickly'),
 Document(id='9c65e98a-e6c3-4753-bc39-54f80b916741', metadata={}, page_content='the most powerful chip ever put in an iPhone... ♪ ...you can push the camera'),
 Document(id='87fa16c0-cfcb-453d-92a8-49b516a359c1', metadata={}, page_content='up to 40 percent faster, unlocked Siri AI, and built the fastest chip'),
 Document(id='e2c9fa13-f555-487f-b1ca-f1ca3ce93a22', metadata={}, page_content='in iPhone history. ♪ [CLICK] Next, you strengthen it... [CLICK]\n[THUD] [HISSING]')]

In [25]:
retriever = vector_store.as_retriever(search_keyword={'k':4})

In [26]:
retriever.invoke('what is feature in iphone 18 pro')

[Document(id='c686c5ed-f36b-4a2b-bfd2-31f213c64774', metadata={}, page_content='to give photographers more creative control. So you add pro controls, allowing them to quickly'),
 Document(id='9c65e98a-e6c3-4753-bc39-54f80b916741', metadata={}, page_content='the most powerful chip ever put in an iPhone... ♪ ...you can push the camera'),
 Document(id='87fa16c0-cfcb-453d-92a8-49b516a359c1', metadata={}, page_content='up to 40 percent faster, unlocked Siri AI, and built the fastest chip'),
 Document(id='e2c9fa13-f555-487f-b1ca-f1ca3ce93a22', metadata={}, page_content='in iPhone history. ♪ [CLICK] Next, you strengthen it... [CLICK]\n[THUD] [HISSING]')]

In [34]:
#Augmentation
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a Helpful assistant. Answer from the following context.
If the context is insufficient then directly say I don't know.
{context}
Question: {question}""",
    input_variables=["context", "question"],
)

In [27]:
question = 'What are the major new features introduced in the iPhone 18 Pro?'

In [28]:
context = retriever.invoke(question)

In [29]:
context

[Document(id='87fa16c0-cfcb-453d-92a8-49b516a359c1', metadata={}, page_content='up to 40 percent faster, unlocked Siri AI, and built the fastest chip'),
 Document(id='9c65e98a-e6c3-4753-bc39-54f80b916741', metadata={}, page_content='the most powerful chip ever put in an iPhone... ♪ ...you can push the camera'),
 Document(id='e2c9fa13-f555-487f-b1ca-f1ca3ce93a22', metadata={}, page_content='in iPhone history. ♪ [CLICK] Next, you strengthen it... [CLICK]\n[THUD] [HISSING]'),
 Document(id='c686c5ed-f36b-4a2b-bfd2-31f213c64774', metadata={}, page_content='to give photographers more creative control. So you add pro controls, allowing them to quickly')]

In [32]:
context = " ".join(i.page_content for i in context)

In [35]:
final_prompt = prompt.invoke({'question': question, 'context': context})

In [44]:
# generation
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2
)

In [46]:
answer = llm.invoke(final_prompt)

In [48]:
print(answer.content)


The iPhone 18 Pro brings several headline‑making upgrades:

| Feature | What it means |
|---------|---------------|
| **Up to 40 % faster performance** | The new SoC delivers noticeably quicker app launches, smoother multitasking and better overall speed. |
| **Unlocked Siri AI** | Siri is now open for developers and users to build on, giving more flexibility and new voice‑assistant possibilities. |
| **Fastest & most powerful chip ever in an iPhone** | The new processor sets a new benchmark for raw performance and efficiency in the iPhone line. |
| **Camera system “pushes the iPhone history”** | Significant camera hardware and software improvements that elevate photo and video quality to a new level. |
| **Pro‑controls for photographers** | Dedicated manual controls (focus, exposure, white‑balance, etc.) allow professional‑grade tweaking directly in the native camera app. |

These are the key highlights mentioned in the announcement.
